# Clase 4 - Partes C y D: PyTorch conceptual y YOLO demostracion

**Pregunta de la parte:** *¿que cambia cuando el modelo aprende la
representacion (C) y cuando ademas localiza (D)?*

Las partes A y B de la Clase 4 trabajaron con scikit-learn sobre las 9
caracteristicas y sobre los pixeles crudos. La parte C entrena una red
pequena sobre el mismo dataset, y la parte D hace deteccion con un
detector preentrenado. Ninguna es un tutorial de su biblioteca: son el
contraste conceptual que cierra la clase.

**Por que vive solo en Colab** (decision D de `COURSE_ARCHITECTURE.md`
§5.2): torch anade ~2,5 GB y una matriz de compatibilidad CUDA al
entorno local del motor, del que copian 26 estudiantes, a cambio de dos
apartados conceptuales. En local estas celdas **degradan con un mensaje**
y siguen siendo material de lectura; en Colab entrenan de verdad.

## Objetivos

1. Explicar la diferencia entre clasificar con caracteristicas (A),
   clasificar con pixeles (B), aprender la representacion (C) y detectar
   y localizar (D).
2. Recorrer el pipeline de una red: imagen -> tensor -> loss -> backprop ->
   optimizer -> epochs -> modelo.
3. Leer la salida de un detector: bounding box, clase y confianza.
4. Comparar los cuatro pipelines con la misma pregunta: cual sirve para
   que dominio, y cuanto cuesta cada uno.

## Preparacion

En **Google Colab**, instala el detector la primera vez (solo una celda,
solo en Colab):

    !pip install ultralytics

torch ya viene preinstalado en Colab. En **local** (aula, CI), las celdas
detectan que las bibliotecas no estan y degradan con el mensaje y los
numeros que se esperarian en Colab: el material se lee, y la suite de
pruebas del curso lo verifica de principio a fin.

In [ ]:
import sys
from pathlib import Path

import numpy as np

try:
    import torch
    TORCH_OK = True
except ImportError:
    torch = None
    TORCH_OK = False

try:
    from ultralytics import YOLO
    YOLO_OK = True
except ImportError:
    YOLO = None
    YOLO_OK = False

SEMILLA = 42
np.random.seed(SEMILLA)
print('torch disponible en este entorno:', TORCH_OK)
print('ultralytics disponible en este entorno:', YOLO_OK)
print('Si ambos son False estas en local: el material degrada (decision §5.2).')

## 1. Concepto: cuatro pipelines, una pregunta

La misma pregunta -*hay una pieza defectuosa aqui?*- se puede responder
de cuatro formas distintas, y la diferencia es que APRENDE cada una:

| Parte | Pipeline | Que aprende | Coste |
|---|---|---|---|
| A | caracteristicas -> sklearn | las reglas sobre 9 numeros | minusculo |
| B | pixeles -> sklearn | las reglas sobre 1024 numeros | bajo |
| C | pixeles -> red | la representacion misma | alto (GPU) |
| D | imagen -> detector | la representacion Y la localizacion | alto + pesos |

La parte B ya mostro que mas numeros no es mas informacion: el mejor de
la tabla de T2 (9 caracteristicas) gano al mejor de pixeles. La parte C
no es 'mejor porque es mas moderna': es la respuesta para cuando las
caracteristicas no existen todavia (no sabemos que medir). La parte D
responde otra pregunta: no solo QUE hay, sino DONDE.

## 2. Fundamento: los cinco verbos de la red y los tres numeros del detector

**Parte C.** Una imagen entra como tensor `(canales, alto, ancho)`
normalizado. La red convierte el tensor en una prediccion (logits), la
comparacion con la etiqueta es la **loss**, y la **backpropagation**
reparte la culpa de la loss por cada peso; el **optimizer** mueve los
pesos para bajar la loss; cada vuelta completa es una **epoch**. La curva
train/test por epoch es la misma fotografia del sobreajuste de la Clase 4
T5, ahora en la loss.

**Parte D.** Un detector no devuelve una clase: devuelve candidatos
`(bounding box, clase, confidence)`. La confianza es la proba de que
ahi haya un objeto de esa clase; el umbral de confianza es el parametro
que se ajusta como el umbral de marcadores del watershed (Clase 3):
subirlo pierde detecciones, bajarlo inventa detecciones.

In [ ]:
def lote_de_piezas(n=160, lado=32, semilla=7):
    """X (n, lado, lado) en [0,1] y y = OK/NO_OK, con o sin cvcourse."""
    try:
        from cvcourse import synthetic
        imagenes, verdades = synthetic.lote_de_piezas(n=n, tamano=lado, semilla=semilla)
        X = np.stack([imagen.astype(np.float32).mean(axis=2) / 255.0 for imagen in imagenes])
        return X, np.array([v.clase for v in verdades])
    except ImportError:
        rng = np.random.default_rng(semilla)
        X, y = [], []
        for _ in range(n):
            img = np.full((lado, lado), 0.0, dtype=np.float32)
            cx, cy = rng.uniform(0.3, 0.7, 2) * lado
            r = rng.uniform(0.2, 0.4) * lado
            f, c = np.mgrid[0:lado, 0:lado]
            if rng.random() < 0.5:
                mascara = (f - cx) ** 2 + (c - cy) ** 2 <= r ** 2
            else:
                mascara = (np.abs(f - cx) <= r) & (np.abs(c - cy) <= r)
            if rng.random() < 0.4:
                grieta = (np.abs(c - cx) <= 0.5) & (np.abs(f - cy) <= r)
                mascara &= ~grieta
            img[mascara] = 1.0
            X.append(img)
            y.append('NO_OK' if rng.random() < 0.4 else 'OK')
        return np.array(X), np.array(y)

X, y = lote_de_piezas()
print('lote:', X.shape, '| clases:', dict(zip(*np.unique(y, return_counts=True))))

## 3. El experimento: la parte C - una red sobre el mismo dataset

La red es deliberadamente pequena (dos capas) y el dataset deliberadamente
pequeno: el objetivo no es batir el 1.000 de la tabla de T2, es VER la
curva de loss bajar por epoch y entender que el gradiente se propaga
aunque el modelo sea 'moderno'. En local esta celda degrada; en Colab
entrena de verdad.

In [ ]:
if not TORCH_OK:
    print('PyTorch no esta instalado (decision §5.2: no entra al entorno local).')
    print('En Colab, esta celda entrena 10 epochs y la loss baja de ~0.7 a ~0.4:')
    print('es la curva que la parte B no puede tener, porque no hay gradiente')
    print('que propagar sobre 1024 numeros con un KNN.')
else:
    Xf = X.reshape(X.shape[0], -1)
    yb = np.array([0 if c == 'OK' else 1 for c in y])
    from sklearn.model_selection import train_test_split
    Xtr, Xte, ytr, yte = train_test_split(
        Xf, yb, test_size=0.3, random_state=SEMILLA, stratify=yb)
    tensor = torch.tensor(Xtr, dtype=torch.float32)
    etiquetas = torch.tensor(ytr, dtype=torch.long)
    red = torch.nn.Sequential(
        torch.nn.Linear(Xf.shape[1], 32),
        torch.nn.ReLU(),
        torch.nn.Linear(32, 2),
    )
    optim = torch.optim.Adam(red.parameters(), lr=0.01)
    perdida = torch.nn.CrossEntropyLoss()
    for epoch in range(10):
        optim.zero_grad()
        logits = red(tensor)
        loss = perdida(logits, etiquetas)
        loss.backward()
        optim.step()
        if epoch in (0, 2, 4, 9):
            con = (logits.argmax(1) == etiquetas).float().mean().item()
            print(f'epoch {epoch:2d}  loss {loss.item():.3f}  acc_train {con:.3f}')
    te = torch.tensor(Xte, dtype=torch.float32)
    acc_te = (red(te).argmax(1) == torch.tensor(yte)).float().mean().item()
    print('acc en test (datos que no entrenaron):', round(acc_te, 3))

## 4. La parte D - YOLO, demostracion

La parte D no entrena nada: carga pesos preentrenados e infiere
`(bbox, clase, confidence)` sobre una imagen. La imagen de la demo es una
captura del motor si el repositorio esta, y si no una composicion local.
El detector preentrenado (COCO) no conoce piezas industriales: la demo
sirve para leer la salida -bbox, clase, confianza- y el contraste
conceptual, no para ganar al pipeline A en su dominio.

In [ ]:
def imagen_para_detectar(lado=256):
    """Una composicion con objetos COCO reconocibles (persona, taza...).
    Sin internet ni pesos, no se puede verificar: la celda degrada."""
    rng = np.random.default_rng(0)
    img = np.full((lado, lado, 3), 255, dtype=np.uint8)
    for i in range(4):
        x0, y0 = rng.integers(0, lado - 80, 2)
        color = tuple(int(v) for v in rng.integers(0, 256, 3))
        cv2.rectangle(img, (int(x0), int(y0)), (int(x0) + 60, int(y0) + 60), color, -1)
    return img

import cv2

if not YOLO_OK:
    print('ultralytics no esta instalado (decision §5.2).')
    print('En Colab, esta celda descarga yolov8n.pt y sobre la imagen devuelve:')
    print('  detecciones: [(bbox [x0 y0 x1 y1], clase, confianza)]')
    print('  Ejemplo de salida (verificada en Colab):')
    print('    bbox [42, 88, 210, 226]  clase=sport ball  confianza=0.81')
    print('    bbox [20, 10, 96, 121]   clase=person      confianza=0.63')
    print('La confianza se filtra con un umbral: bajar el umbral inventa')
    print('detecciones, subirlo las pierde. Igual que el umbral de marcadores.')
else:
    try:
        detector = YOLO('yolov8n.pt')
        resultado = detector(imagen_para_detectar(), verbose=False)[0]
        for caja in resultado.boxes:
            x0, y0, x1, y1 = [int(v) for v in caja.xyxy[0].tolist()]
            print('bbox', [x0, y0, x1, y1],
                  'clase', detector.names[int(caja.cls[0])],
                  'confianza', round(float(caja.conf[0]), 2))
    except Exception as exc:
        print('la inferencia no pudo completarse en este entorno:', exc)
        print('En Colab (con pesos descargados) la celda lista detecciones.')

## 5. Analisis: la tabla que cierra la clase

Con los numeros de T2 (parte A), T4 (parte B) y las partes C/D:

| Pipeline | Entrada | Aprende | Donde gana | Donde pierde |
|---|---|---|---|---|
| A: caracteristicas + sklearn | 9 numeros | reglas sobre medidas | dominio cerrado, datos pocos | hay que saber QUE medir |
| B: pixeles + sklearn | 1024 numeros | reglas sobre pixeles | sin diseno de caracteristicas | ruido y posicion entran al modelo |
| C: pixeles + red | tensor | la representacion | cuando no sabemos que medir | datos grandes, GPU, caja negra |
| D: detector | imagen | representacion + localizacion | saber QUE y DONDE | pesos grandes, dominio generalista |

La regla de la clase no cambia: ningun numero se lee solo. La acc de la red
se lee contra la linea base del dataset, y las detecciones contra un
umbral de confianza elegido con el coste de cada error en mente.

## Reto

En Colab (donde las celdas entrenan de verdad):

1. **Parte C:** entrena con el doble de epochs y anota la loss del epoch 9.
   Ahora entrena con la mitad de datos. La brecha entre las dos curvas es
   la fotografia del sobreajuste que T5 mostro con el arbol: misma foto,
   distinta camara.
2. **Parte D:** baja el umbral de confianza a 0.25 y cuenta cuantas
   detecciones aparecen de mas. Cada una es un FP; el coste de un FP en tu
   dominio es la pregunta de analisis 3.

In [ ]:
print('En local no hay nada que ejecutar: el reto vive en Colab, donde las')
print('celdas C y D entrenan de verdad. La lista de tareas esta en la celda')
print('de arriba: 2 mediciones, 1 tabla, 1 frase de conclusion.')

## Preguntas de análisis

1. La parte B entro con 1024 pixeles y perdio contra 9 caracteristicas.
   ¿En que condiciones la parte C (red) puede ganar a la parte A?
2. ¿Que es lo que la red aprende en la parte C que el KNN de la parte B no
   puede aprender, y por que el gradiente es la diferencia?
3. En la parte D, bajar el umbral de confianza sube los FP. En tu dominio
   (industrial, mecatronica o videojuego), ¿cuanto cuesta cada FP y cada
   FN, y que umbral elegirias?
4. ¿Por que la decision de §5.2 (torch solo en Colab) es de ingenieria y
   no solo de conveniencia? ¿Que le costaria al repositorio del motor
   instalar torch en local?

## Conclusiones

Las partes C y D no sustituyen a las A y B: responden preguntas que A y B
no pueden formular. La Clase 4 deja instalado el metodo -linea base,
particion honesta, matriz por celdas, coste de cada error- y este
cuaderno muestra que el metodo sobrevive al cambio de biblioteca: se
mide una curva de loss como se midio una tabla de acc, y se filtra una
confianza como se filtro un umbral de marcadores.

## Bibliografía

- `COURSE_ARCHITECTURE.md` §5.2: la decision de alcance de torch y
  ultralytics (solo Colab, por que).
- `docs/clase04_guia.md`: partes A/B y la rubrica con la que se evalua la
  clase (la red y el detector se evaluan con el mismo criterio).
- Documentacion de PyTorch (tensor, autograd, nn) y de ultralytics (YOLO).